# Aula 12 - PySpark: NOT e filtrando dados nulos com funções isNull

In [0]:
# Aula 12 - NOT e filtragem de nulos com isNull()
# Objetivo: distinguir linhas com valor ausente (NULL) das preenchidas

# Imports das funções do PySpark (col, etc.)
from pyspark.sql.functions import *

# Leitura do CSV com header + UTF-8; distinct() elimina duplicatas na carga
df_carros = (
    spark.read
    .format('csv')
    .option('header', 'true')
    .option('encoding', 'utf-8')
    .load('/Workspace/Users/jcesarsantana215@gmail.com/Aulas PySpark/datasets/modelo_carro.csv', sep=',')
    .distinct()
)

# DataFrame auxiliar 'source' (ids 1-3) e 'final' (ids 1-4) para inspeção rápida
df_carros_source = df_carros.where(
    (col("id_carro") == '1') |
    (col("id_carro") == '2') |
    (col("id_carro") == '3')
)

df_carros_final = df_carros.where(
    (col("id_carro") == '1') |
    (col("id_carro") == '2') |
    (col("id_carro") == '3') |
    (col("id_carro") == '4')
)

display(df_carros_source)

display(df_carros_final)

In [0]:
# Registra o DataFrame como view 'carros' e filtra nulos via SQL
# 'IS NULL' no Spark captura NULL real (não string vazia nem 'null' textual)
# Aqui buscamos carros sem preço preenchido
df_carros.createOrReplaceTempView("carros")

df_carros_sql = spark.sql("""
SELECT * FROM carros
WHERE preco IS NULL
""")

display(df_carros_sql)

In [0]:
%sql
-- Mesma consulta IS NULL usando o magic %sql do Databricks
-- Forma rápida de prototipar; o resultado aparece como tabela, mas não vira DataFrame
SELECT * FROM carros
WHERE preco IS NULL

In [0]:
# Filtrar nulos com a API PySpark: col('preco').isNull()
# Equivalente ao 'WHERE preco IS NULL' do SQL
# Retorna só as linhas onde o preço está ausente
df_carros_spark = df_carros.where(
    col('preco').isNull()
)

display(df_carros_spark)

In [0]:
# Filtrar NÃO nulos negando com o operador ~ (negação no PySpark)
# ~col('preco').isNull() equivale a 'WHERE preco IS NOT NULL'
# Retorna só as linhas com preço preenchido — o complemento da célula anterior
df_carros_spark = df_carros.where(
    ~col('preco').isNull()
)

display(df_carros_spark)